<a href="https://colab.research.google.com/github/lucaser9898/sovereign-funds-rl-portfolio/blob/main/Definitivo_Q_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Q-Learning Portfolio
# ----------------------------------------------------------------------------------------------

from __future__ import annotations
import os, math, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Optional
from matplotlib.colors import ListedColormap, BoundaryNorm

# =========================
# PARAMETRI (Colab widgets)
# =========================
# RL / Exploration
EPOCHS    = 100        #@param {type:"integer"}
ALPHA     = 0.02       #@param {type:"number"}
GAMMA     = 0.90       #@param {type:"number"}
EPS_START = 0.20       #@param {type:"number"}
EPS_END   = 0.05       #@param {type:"number"}
EPS_DECAY = 0.98       #@param {type:"number"}
L2_REG    = 1e-4       #@param {type:"number"}

# Action space / Environment
TILT_MIN  = 0.4        #@param {type:"number"}
TILT_MAX  = 1.6        #@param {type:"number"}
TILT_STEP = 0.2       #@param {type:"number"}
RISK_AV   = 0.5        #@param {type:"number"}
COST_RATE = 0.0001     #@param {type:"number"}
COV_WIN   = 12         #@param {type:"integer"}
SEED      = 125        #@param {type:"integer"}

# Evaluation
EVAL_MODE       = "split"   #@param ["split","walkforward"] {allow-input: true}
TRAIN_RATIO     = 0.5             #@param {type:"number"}
WF_TRAIN_YEARS  = 3               #@param {type:"integer"}
WF_TEST_MONTHS  = 6               #@param {type:"integer"}

# Features (mensili)
FEAT_SHORT_WIN  = 3               #@param {type:"integer"}
FEAT_LONG_WIN   = 6               #@param {type:"integer"}

# Metrics frequency (12 = mensile)
METRICS_FREQ = 12                 #@param {type:"integer"}

# Over/Under-weight matrix
MATRIX_MODE = "ratio"             #@param ["ratio","diff"] {allow-input: true}
MATRIX_TOL  = 0.01                #@param {type:"number"}
MATRIX_AGG  = "Y"                 #@param ["None","M","Q","Y","BINS"] {allow-input: true}
N_BUCKETS   = 20                  #@param {type:"integer"}

# Compact plot
ORDER_BY_ACTIVITY = True          #@param {type:"boolean"}
TOP_K_ASSETS      = 40            #@param {type:"integer"}
LAST_N_PERIODS    = 240           #@param {type:"integer"}
Y_FONTSIZE        = 8             #@param {type:"integer"}
X_ROTATION        = 45            #@param {type:"integer"}

# I/O
OUT_DIR = "qlearning_outputs"     #@param {type:"string"}

# =========================
# 0) DATA LOADING (prezzi mensili)
# =========================
try:
    from google.colab import files  # type: ignore
    USE_COLAB = True
except Exception:
    USE_COLAB = False

def _match_file(base_prefix: str, uploaded: dict) -> str:
    for k in uploaded.keys():
        if k.lower().startswith(base_prefix.lower()):
            return k
    raise FileNotFoundError(f"File starting with '{base_prefix}' not found. Uploaded: {list(uploaded.keys())}")

if USE_COLAB:
    print("Upload: indici_borsa_filtrati.csv  e  pesi_ritracciati.csv")
    uploaded = files.upload()
    PATH_INDICI = _match_file("indici_borsa_filtrati", uploaded)
    PATH_PESI   = _match_file("pesi_ritracciati", uploaded)
    print("Using:", PATH_INDICI, "and", PATH_PESI)
else:
    PATH_INDICI = "indici_borsa_filtrati.csv"
    PATH_PESI   = "pesi_ritracciati.csv"

def read_csv_infer_datetime(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    date_cols = [c for c in df.columns if c.lower() in ("date","data","timestamp")]
    if date_cols:
        dcol = date_cols[0]
        df[dcol] = pd.to_datetime(df[dcol], errors="coerce", dayfirst=True)
        df = df.dropna(subset=[dcol]).sort_values(dcol).set_index(dcol)
    else:
        df.index = pd.RangeIndex(len(df))
    df = df.dropna(axis=1, how="all")
    return df

raw_prices = read_csv_infer_datetime(PATH_INDICI)
raw_pesi   = read_csv_infer_datetime(PATH_PESI)

def to_arith_returns_from_prices(df_prices: pd.DataFrame) -> pd.DataFrame:
    df = df_prices.select_dtypes(include=[np.number]).copy()
    if df.empty:
        raise ValueError("Nessuna colonna numerica trovata nei prezzi.")
    # r_t = P_t/P_{t-1} - 1
    rets = df.pct_change()
    return rets.replace([np.inf, -np.inf], np.nan).dropna(how="all")

def parse_base_weights(raw_weights: pd.DataFrame, asset_cols: List[str]) -> pd.DataFrame:
    """
    Accetta:
    - Statico: ['Asset','Peso']
    - Long:    ['Date/Data','Asset','Peso']
    - Wide:    indice datetime + colonne = asset
    """
    df = raw_weights.copy()
    cols_lower = {c.lower(): c for c in df.columns}

    if "asset" in cols_lower and "peso" in cols_lower and not (("date" in cols_lower) or ("data" in cols_lower)):
        asset_col = cols_lower["asset"]; peso_col = cols_lower["peso"]
        s = df[[asset_col, peso_col]].dropna().groupby(asset_col)[peso_col].last()
        s = s.reindex(asset_cols).fillna(0.0)
        if s.sum() <= 0: s = pd.Series(1.0, index=asset_cols)
        s = s / s.sum()
        return pd.DataFrame([s.values], columns=s.index, index=[pd.Timestamp("1970-01-01")])

    if "peso" in cols_lower and ("asset" in cols_lower) and (("date" in cols_lower) or ("data" in cols_lower)):
        asset_col = cols_lower["asset"]; peso_col = cols_lower["peso"]
        date_col = cols_lower.get("date", cols_lower.get("data"))
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce", dayfirst=True)
        df = df.dropna(subset=[date_col, asset_col, peso_col])
        piv = df.pivot_table(index=date_col, columns=asset_col, values=peso_col, aggfunc="last")
        piv = piv.reindex(columns=asset_cols).sort_index().ffill().fillna(0.0)
        row_sums = piv.sum(axis=1).replace(0, np.nan)
        return piv.div(row_sums, axis=0).fillna(1.0/len(asset_cols))

    df_wide = df.select_dtypes(include=[np.number]).copy()
    if set(asset_cols).issubset(df_wide.columns):
        piv = df_wide.reindex(columns=asset_cols).sort_index().ffill().fillna(0.0)
        row_sums = piv.sum(axis=1).replace(0, np.nan)
        return piv.div(row_sums, axis=0).fillna(1.0/len(asset_cols))

    return pd.DataFrame([np.repeat(1.0/len(asset_cols), len(asset_cols))],
                        columns=asset_cols, index=[pd.Timestamp("1970-01-01")])

rets_all = to_arith_returns_from_prices(raw_prices).sort_index().dropna(how="all")
assets = list(rets_all.columns)
assert len(assets) >= 2, "Servono almeno 2 asset."

base_weights_df = parse_base_weights(raw_pesi, assets)
start_date = max(rets_all.index.min(), base_weights_df.index.min()) if len(base_weights_df) > 1 else rets_all.index.min()
rets_all = rets_all[rets_all.index >= start_date].copy()
if len(base_weights_df) == 1:
    base_weights = pd.DataFrame(np.repeat(base_weights_df.values, len(rets_all), axis=0),
                                index=rets_all.index, columns=assets)
else:
    base_weights = base_weights_df.reindex(rets_all.index, method="ffill").ffill()
    base_weights = base_weights.div(base_weights.sum(axis=1).replace(0, np.nan), axis=0).fillna(1.0/len(assets))

_eps = 1e-9
base_weights = base_weights.clip(lower=_eps)
base_weights = base_weights.div(base_weights.sum(axis=1), axis=0)

# =========================
# 1) FEATURES (past-only)
# =========================
def compute_features(rets: pd.DataFrame, assets: List[str],
                     win_short: int, win_long: int) -> Dict[str, pd.DataFrame]:
    mom_s = rets.rolling(win_short, min_periods=win_short).mean()
    mom_l = rets.rolling(win_long,  min_periods=win_long ).mean()
    vol_s = rets.rolling(win_short, min_periods=win_short).std().replace(0, np.nan)
    global_mom_s = mom_s.mean(axis=1).to_frame("global_momS")
    global_mom_l = mom_l.mean(axis=1).to_frame("global_momL")
    market_vol   = vol_s.mean(axis=1).to_frame("market_volS")
    ranks = mom_s.rank(axis=1, method="average")
    q = max(1, int(len(assets)*0.25))
    top_mask = ranks.apply(lambda row: row >= (len(assets)-q+1), axis=1)
    bot_mask = ranks.apply(lambda row: row <= q, axis=1)
    top_mean = mom_s.where(top_mask).mean(axis=1)
    bot_mean = mom_s.where(bot_mask).mean(axis=1)
    spread = (top_mean - bot_mean).to_frame("momS_spread")
    return {
        "momS": mom_s, "momL": mom_l, "volS": vol_s,
        "global": pd.concat([global_mom_s, global_mom_l, market_vol, spread], axis=1)
    }

features = compute_features(rets_all, assets, FEAT_SHORT_WIN, FEAT_LONG_WIN)

# =========================
# 2) RISK MODEL — EWMA covariance (half-life standard)
# =========================
def ewma_cov(returns_window: pd.DataFrame, half_life: int) -> np.ndarray:
    X = returns_window.values
    T, N = X.shape
    if T < 5:
        v = np.nanvar(X, axis=0)
        m = np.nanmean(v) if np.isfinite(v).any() else 0.0
        v = np.nan_to_num(v, nan=m)
        return np.diag(v)
    lam = float(np.exp(-math.log(2.0) / max(1.0, half_life)))  # decay per step
    a = 1.0 - lam
    Xc = X - np.nanmean(X, axis=0, keepdims=True)
    S = np.zeros((N,N)); w = 0.0
    for t in range(T-1, -1, -1):
        xt = np.nan_to_num(Xc[t], nan=0.0)
        S = lam*S + a*np.outer(xt, xt)
        w = lam*w + a
    if w > 0:
        S = S / w
    # shrink diagonale leggera
    diag = np.diag(np.diag(S))
    S = 0.9*S + 0.1*diag
    return S

# =========================
# 3) AGENTE
# =========================
class QLearnerFA:
    def __init__(self, n_actions: int, n_features: int,
                 alpha: float=0.02, gamma: float=0.90,
                 eps_start: float=0.20, eps_end: float=0.05, eps_decay: float=0.98,
                 l2: float=1e-4, seed: int=123):
        rng = np.random.default_rng(seed)
        self.n_actions = n_actions
        self.n_features = n_features
        self.theta = rng.normal(scale=0.01, size=(n_actions, n_features))
        self.alpha = alpha
        self.gamma = gamma
        self.eps = eps_start
        self.eps_end = eps_end
        self.eps_decay = eps_decay
        self.l2 = l2
        self.rng = rng

    def q_sa(self, a: int, phi: np.ndarray) -> float:
        return float(self.theta[a].dot(phi))

    def select_action(self, phis: List[np.ndarray]) -> int:
        if self.rng.random() < self.eps:
            return int(self.rng.integers(0, len(phis)))
        q_vals = np.array([self.q_sa(a, phis[a]) for a in range(len(phis))])
        return int(np.argmax(q_vals))

    def update(self, phi_sa: np.ndarray, a: int, reward: float, max_q_next: float | None):
        # weight decay L2
        self.theta[a] *= (1.0 - self.alpha * self.l2)
        q_sa = self.q_sa(a, phi_sa)
        target = reward if max_q_next is None else reward + self.gamma * max_q_next
        self.theta[a] += self.alpha * (target - q_sa) * phi_sa

    def decay_epsilon(self):
        self.eps = max(self.eps_end, self.eps * self.eps_decay)

# =========================
# 4) ENV — timing corretto, feature scaling, costi su tilt
# =========================
class PortfolioEnv:
    """
    Azioni: per ciascun asset {+tilt, -tilt} + HOLD.
    Decidi a t (features fino a t). I pesi valgono per t→t+1.
    Reward = r_{t+1} - RISK_AV * vol_t - COST_RATE * turnover_tilt.
    """
    def __init__(self, rets: pd.DataFrame, base_w: pd.DataFrame, feats: Dict[str, pd.DataFrame],
                 tilt_min: float=0.6, tilt_max: float=1.4, tilt_step: float=0.1,
                 risk_aversion: float=1.0, cost_rate: float=0.0002, cov_half_life: int=63,
                 slice_index: Optional[pd.Index]=None):
        assert rets.index.equals(base_w.index), "Returns e base weights devono avere lo stesso indice."
        self.rets_all = rets
        self.base_all = base_w
        self.assets = list(rets.columns)
        self.n_assets = len(self.assets)
        self.feats = feats
        self.cost_rate = cost_rate
        self.risk_aversion = risk_aversion
        self.cov_hl = cov_half_life
        self.tilt_min = tilt_min
        self.tilt_max = tilt_max
        self.tilt_step = tilt_step
        self.idx = slice_index if slice_index is not None else rets.index

        g = self.feats["global"].dropna().index
        ms = self.feats["momS"].dropna().index
        ml = self.feats["momL"].dropna().index
        vs = self.feats["volS"].dropna().index
        first_feat_date = max(d[0] for d in [g, ms, ml, vs])
        valid = self.idx[self.idx >= first_feat_date]
        if len(valid) < 2:
            raise ValueError("Finestra troppo corta per iniziare.")
        self.sub_index = valid
        self.first_t_pos = 0
        self.last_t_pos  = len(valid) - 2
        self.reset()

    @property
    def n_actions(self) -> int: return 2*self.n_assets + 1
    @property
    def n_features(self) -> int: return 10  # 5 global + 5 asset

    def reset(self) -> int:
        self.t_pos = self.first_t_pos
        self.tilts = np.ones(self.n_assets)
        t0 = self.sub_index[self.t_pos]
        self.base_row_t = self.base_all.loc[t0].values
        self.weights_t = self._compute_weights(self.tilts, self.base_row_t)
        return self.t_pos

    def _compute_weights(self, tilts: np.ndarray, base_row: np.ndarray) -> np.ndarray:
        w_raw = np.clip(base_row * tilts, 1e-12, None)
        return w_raw / np.sum(w_raw)

    # ---- feature helpers (con tanh-scaling) ----
    def _global_features(self, pos: int) -> np.ndarray:
        tstamp = self.sub_index[pos]
        g = self.feats["global"].loc[tstamp]
        arr = np.array([1.0, g["global_momS"], g["global_momL"], g["market_volS"], g["momS_spread"]], dtype=float)
        return np.tanh(arr)  # bounding

    def _asset_features(self, asset_idx: int, pos: int) -> np.ndarray:
        tstamp = self.sub_index[pos]
        a = self.assets[asset_idx]
        arr = np.array([
            self.feats["momS"].loc[tstamp][a],
            self.feats["momL"].loc[tstamp][a],
            self.feats["volS"].loc[tstamp][a],
            self.tilts[asset_idx],
            self.weights_t[asset_idx]
        ], dtype=float)
        return np.tanh(arr)  # bounding

    def phi(self, action: int, pos: int) -> np.ndarray:
        base = self._global_features(pos)
        if action == 2*self.n_assets:
            asset_f = np.zeros(5, dtype=float)  # HOLD
        else:
            asset_f = self._asset_features(action // 2, pos)
        phi = np.concatenate([base, asset_f], axis=0)
        return phi

    def step(self, action: int):
        done = False
        tstamp_t = self.sub_index[self.t_pos]
        base_row_t = self.base_all.loc[tstamp_t].values

        # Pesi prima dell'azione (stesso base di t) per calcolare turnover da tilt
        weights_before = self._compute_weights(self.tilts, base_row_t)

        # 1) aggiorna tilts
        if action != 2*self.n_assets:
            asset_idx = action // 2
            direction = +1 if (action % 2 == 0) else -1
            self.tilts[asset_idx] = float(np.clip(self.tilts[asset_idx] + direction*self.tilt_step,
                                                  self.tilt_min, self.tilt_max))

        # 2) pesi per t→t+1 usando base a t
        weights_next = self._compute_weights(self.tilts, base_row_t)

        # 3) rischio a t (EWMA su dati ≤ t)
        all_idx = self.rets_all.index
        t_global_loc = int(np.searchsorted(all_idx.values, tstamp_t))
        start = max(0, t_global_loc - max(20, self.cov_hl*2))
        window = self.rets_all.iloc[start:t_global_loc+1]
        Sigma = ewma_cov(window, half_life=self.cov_hl)
        vol_est = float(np.sqrt(max(0.0, weights_next @ Sigma @ weights_next.T)))

        # 4) payoff realizzato con r_{t+1}
        tstamp_tp1 = self.sub_index[self.t_pos+1]
        r_vec_tp1 = self.rets_all.loc[tstamp_tp1].values
        port_ret = float(np.dot(weights_next, r_vec_tp1))

        # 5) costi / turnover da tilt (stesso base)
        turnover_tilt = float(np.sum(np.abs(weights_next - weights_before)))
        cost = self.cost_rate * turnover_tilt
        reward = port_ret - self.risk_aversion * vol_est - cost

        # 6) advance time
        self.t_pos += 1
        self.weights_t = self._compute_weights(self.tilts, self.base_all.loc[self.sub_index[self.t_pos]].values)
        if self.t_pos >= self.last_t_pos:
            done = True

        info = {
            "t_end": self.sub_index[self.t_pos],
            "weights_next": weights_next.copy(),
            "tilts": self.tilts.copy(),
            "port_ret": port_ret, "vol_est": vol_est, "turnover_tilt": turnover_tilt, "cost": cost
        }
        return self.t_pos, reward, done, info

# =========================
# 5) TRAIN / TEST ROUTINES
# =========================
def train_agent(env: PortfolioEnv, agent: QLearnerFA, epochs: int):
    for _ in range(epochs):
        pos = env.reset()
        done = False
        while not done:
            phis = [env.phi(a, pos) for a in range(env.n_actions)]
            a = agent.select_action(phis)
            pos_next, reward, done, _ = env.step(a)
            if not done:
                phis_next = [env.phi(a2, pos_next) for a2 in range(env.n_actions)]
                q_next_vals = np.array([agent.q_sa(a2, phis_next[a2]) for a2 in range(env.n_actions)])
                agent.update(phis[a], a, reward, float(np.max(q_next_vals)))
                pos = pos_next
            else:
                agent.update(phis[a], a, reward, None)
        agent.decay_epsilon()

def greedy_backtest(env: PortfolioEnv, agent: QLearnerFA):
    pos = env.reset(); done=False
    dates, actions = [], []
    port_rets, vols, costs, turns = [], [], [], []
    weights_list, tilts_list = [], []

    while not done:
        phis = [env.phi(a, pos) for a in range(env.n_actions)]
        q_vals = np.array([agent.q_sa(a, phis[a]) for a in range(env.n_actions)])
        a = int(np.argmax(q_vals))
        pos_next, _, done, info = env.step(a)

        dates.append(info["t_end"]); actions.append(a)
        port_rets.append(info["port_ret"]); vols.append(info["vol_est"])
        costs.append(info["cost"]); turns.append(info["turnover_tilt"])
        weights_list.append(info["weights_next"]); tilts_list.append(info["tilts"])

        pos = pos_next

    weights_df = pd.DataFrame(weights_list, index=dates, columns=env.assets)
    tilts_df   = pd.DataFrame(tilts_list,   index=dates, columns=env.assets)
    actions_ser   = pd.Series(actions, index=dates, name="action")
    port_rets_ser = pd.Series(port_rets, index=dates, name="portfolio_return")
    aux = {"vol": pd.Series(vols, index=dates), "cost": pd.Series(costs, index=dates), "turnover": pd.Series(turns, index=dates)}
    return weights_df, actions_ser, port_rets_ser, tilts_df, aux

# =========================
# 6) EVAL: SPLIT o WALK-FORWARD
# =========================
np.random.seed(SEED)

if EVAL_MODE.lower() == "split":
    idx = rets_all.index
    split = int(len(idx) * TRAIN_RATIO)
    train_idx = idx[:split]
    test_idx  = idx[max(0, split-1):]

    env_tr = PortfolioEnv(rets_all, base_weights, features, TILT_MIN, TILT_MAX, TILT_STEP,
                          RISK_AV, COST_RATE, COV_WIN, slice_index=train_idx)
    agent  = QLearnerFA(env_tr.n_actions, env_tr.n_features, ALPHA, GAMMA, EPS_START, EPS_END, EPS_DECAY, L2_REG, SEED)
    train_agent(env_tr, agent, EPOCHS)

    env_te = PortfolioEnv(rets_all, base_weights, features, TILT_MIN, TILT_MAX, TILT_STEP,
                          RISK_AV, COST_RATE, COV_WIN, slice_index=test_idx)
    weights_df, actions_ser, port_rets_ser, tilts_df, aux = greedy_backtest(env_te, agent)

else:
    idx_all = rets_all.index
    step = int(WF_TEST_MONTHS * 1)        # ~ 1 riga = 1 mese
    win  = int(WF_TRAIN_YEARS * 12)       # mesi in training window

    # ensure features available
    g0 = features["global"].dropna().index[0]
    start_pos = max(win, int(np.searchsorted(idx_all.values, g0)))

    w_list, a_list, r_list, t_list, v_list, c_list, u_list = [], [], [], [], [], [], []

    pos = start_pos
    while pos < len(idx_all)-2:
        train_slice = idx_all[max(0, pos-win):pos]
        test_slice  = idx_all[pos-1:min(len(idx_all), pos+step)]

        if len(train_slice) < max(24, FEAT_LONG_WIN+2) or len(test_slice) < 2:
            break

        env_tr = PortfolioEnv(rets_all, base_weights, features, TILT_MIN, TILT_MAX, TILT_STEP,
                              RISK_AV, COST_RATE, COV_WIN, slice_index=train_slice)
        agent  = QLearnerFA(env_tr.n_actions, env_tr.n_features, ALPHA, GAMMA, EPS_START, EPS_END, EPS_DECAY, L2_REG, SEED)
        train_agent(env_tr, agent, EPOCHS)

        env_te = PortfolioEnv(rets_all, base_weights, features, TILT_MIN, TILT_MAX, TILT_STEP,
                              RISK_AV, COST_RATE, COV_WIN, slice_index=test_slice)
        w,a,r,t,aux_b = greedy_backtest(env_te, agent)

        w_list.append(w); a_list.append(a); r_list.append(r); t_list.append(t)
        v_list.append(aux_b["vol"]); c_list.append(aux_b["cost"]); u_list.append(aux_b["turnover"])

        pos += step

    if len(r_list) == 0:
        raise RuntimeError("Walk-forward senza blocchi validi. Controlla span e parametri.")

    weights_df   = pd.concat(w_list).sort_index()
    actions_ser  = pd.concat(a_list).sort_index()
    port_rets_ser= pd.concat(r_list).sort_index()
    tilts_df     = pd.concat(t_list).sort_index()
    aux = {"vol": pd.concat(v_list).sort_index(),
           "cost": pd.concat(c_list).sort_index(),
           "turnover": pd.concat(u_list).sort_index()}

# =========================
# 7) PLOTS & METRICHE (freq=12)
# =========================
def performance_metrics(returns: pd.Series, freq: int=12) -> Dict[str, float]:
    rets = returns.dropna()
    n = len(rets)
    if n == 0:
        return {"CAGR": np.nan, "Vol_ann": np.nan, "Sharpe": np.nan, "MaxDD": np.nan}
    eq = (1+rets).cumprod()
    years = n / freq
    cagr = (eq.iloc[-1])**(1/years) - 1 if years > 0 else np.nan
    vol = rets.std() * math.sqrt(freq) if n > 1 else np.nan
    sharpe = (rets.mean() * freq) / vol if (vol and vol > 0) else np.nan
    maxdd = (eq/eq.cummax() - 1.0).min()
    return {"CAGR": float(cagr), "Vol_ann": float(vol), "Sharpe": float(sharpe), "MaxDD": float(maxdd)}

eq = (1+port_rets_ser.fillna(0)).cumprod()
plt.figure(figsize=(10,4))
plt.plot(eq.index, eq.values)
plt.title(f"Equity Line - DQN-learning OOS ({EVAL_MODE}, EWMA risk)")
plt.xlabel("Date"); plt.ylabel("Cumulative returns")
plt.tight_layout(); plt.show()

metrics = performance_metrics(port_rets_ser, freq=METRICS_FREQ)
print("OOS Metrics (freq=12):")
for k,v in metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) and not np.isnan(v) else f"  {k}: {v}")

# =========================
# 8) OVER/UNDER-WEIGHT MATRIX
# =========================
def _ensure_dtindex(df: pd.DataFrame) -> pd.DataFrame:
    if not isinstance(df.index, pd.DatetimeIndex):
        df = df.copy()
        df.index = pd.to_datetime(df.index, errors="coerce", dayfirst=True)
    df = df[~df.index.isna()].copy()
    df = df[~df.index.duplicated(keep="last")]
    return df.sort_index()

weights_df   = _ensure_dtindex(weights_df)
base_weights = _ensure_dtindex(base_weights)

def over_under_sign_matrix(weights_df: pd.DataFrame,
                           base_weights: pd.DataFrame,
                           mode: str = "ratio",
                           tol: float = 0.01,
                           agg: str | None = None) -> pd.DataFrame:
    base_aligned = base_weights.reindex(weights_df.index).ffill()
    common = [c for c in weights_df.columns if c in base_aligned.columns]
    if len(common) == 0:
        raise ValueError("Nessuna colonna in comune tra weights e base_weights")
    W = weights_df[common].copy()
    B = base_aligned[common].copy()

    eps_local = 1e-12
    W = W.clip(lower=0).div(W.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    B = B.clip(lower=eps_local).div(B.sum(axis=1).replace(0, np.nan), axis=0).fillna(eps_local)

    if mode == "ratio":
        M = (W / B) - 1.0
    elif mode == "diff":
        M = W - B
    else:
        raise ValueError("mode deve essere 'ratio' o 'diff'.")

    sign = M.copy()
    sign[(M > -tol) & (M < tol)] = 0.0
    sign = np.sign(sign)

    if agg is not None:
        sign = sign.resample(agg).mean()
        sign = sign.applymap(lambda x: 1 if x > +tol else (-1 if x < -tol else 0))

    return sign.astype(int)

def sign_matrix_n_buckets(sign_df: pd.DataFrame, n: int = 10, tol: float = 0.01) -> pd.DataFrame:
    idx = pd.to_datetime(sign_df.index)
    start, end = idx.min(), idx.max()
    bins = pd.date_range(start=start, end=end + pd.Timedelta(days=1), periods=n+1)
    labels = [f"B{i+1} {bins[i].date()}→{(bins[i+1]-pd.Timedelta(days=1)).date()}" for i in range(n)]
    cats = pd.cut(idx, bins=bins, labels=labels, include_lowest=True, right=False)
    grouped = sign_df.groupby(cats).mean()
    grouped = grouped.applymap(lambda x: 1 if x > +tol else (-1 if x < -tol else 0))
    return grouped

def plot_over_under_sign(
    sign_df: pd.DataFrame,
    title: str = "Over/Under-Weight vs Base (compact)",
    order_by_activity: bool = True,
    top_k_assets: int | None = 44,
    last_n_periods: int | None = 10,
    y_fontsize: int = 8,
    x_rotation: int = 45
):
    M = sign_df.copy()
    if last_n_periods is not None and last_n_periods > 0:
        M = M.iloc[-last_n_periods:, :]
    if order_by_activity:
        order = M.abs().mean(axis=0).sort_values(ascending=False).index
        M = M[order]
    if top_k_assets is not None and top_k_assets < M.shape[1]:
        M = M.iloc[:, :top_k_assets]

    cmap = ListedColormap(["#2c7fb8", "#ffffff", "#d7301f"])  # blue, white, red
    bounds = [-1.5, -0.5, 0.5, 1.5]
    norm = BoundaryNorm(bounds, cmap.N)

    n_periods, n_assets = M.shape[0], M.shape[1]
    fig_w = min(16, max(6, 0.6 * n_periods))
    fig_h = min(14, max(4, 0.30 * n_assets))

    plt.figure(figsize=(fig_w, fig_h))
    plt.imshow(M.T.values, aspect="auto", cmap=cmap, norm=norm)
    plt.yticks(range(n_assets), M.columns, fontsize=y_fontsize)
    try:
        xticks = [pd.to_datetime(d).date() for d in M.index]
    except Exception:
        xticks = list(M.index)
    plt.xticks(range(n_periods), xticks, rotation=x_rotation)
    plt.xlabel("Periodo")
    plt.title(title)
    cbar = plt.colorbar(ticks=[-1, 0, 1])
    cbar.ax.set_yticklabels(["Under", "Neutral", "Over"])
    plt.tight_layout()
    plt.show()

agg_opt = None if (str(MATRIX_AGG).lower()=="none") else (None if str(MATRIX_AGG).upper()=="BINS" else MATRIX_AGG)
sign_base = over_under_sign_matrix(weights_df, base_weights, mode=MATRIX_MODE, tol=MATRIX_TOL, agg=agg_opt)
sign_view = sign_matrix_n_buckets(sign_base, n=max(1, int(N_BUCKETS)), tol=MATRIX_TOL) if str(MATRIX_AGG).upper() == "BINS" else sign_base

plot_over_under_sign(
    sign_view,
    title=f"Over/Under-Weight (agg={MATRIX_AGG}, top {TOP_K_ASSETS}, last {LAST_N_PERIODS})",
    order_by_activity=ORDER_BY_ACTIVITY,
    top_k_assets=(None if TOP_K_ASSETS<=0 else TOP_K_ASSETS),
    last_n_periods=(None if LAST_N_PERIODS<=0 else LAST_N_PERIODS),
    y_fontsize=Y_FONTSIZE,
    x_rotation=X_ROTATION
)

# =========================
# 9) SAVE OUTPUTS
# =========================
os.makedirs(OUT_DIR, exist_ok=True)
paths = {
    "weights_csv": os.path.join(OUT_DIR, f"weights_rl_OOS_{EVAL_MODE}.csv"),
    "tilts_csv": os.path.join(OUT_DIR, f"tilts_rl_OOS_{EVAL_MODE}.csv"),
    "actions_csv": os.path.join(OUT_DIR, f"actions_rl_OOS_{EVAL_MODE}.csv"),
    "portfolio_returns_csv": os.path.join(OUT_DIR, f"portfolio_returns_rl_OOS_{EVAL_MODE}.csv"),
    "metrics_json": os.path.join(OUT_DIR, f"metrics_OOS_{EVAL_MODE}.json"),
    "matrix_csv": os.path.join(OUT_DIR, f"overunder_matrix_{MATRIX_AGG}.csv"),
}
weights_df.to_csv(paths["weights_csv"])
tilts_df.to_csv(paths["tilts_csv"])
actions_ser.to_csv(paths["actions_csv"])
port_rets_ser.to_csv(paths["portfolio_returns_csv"])
with open(paths["metrics_json"], "w") as f: json.dump({
    **metrics, "freq_used": METRICS_FREQ, "eval_mode": EVAL_MODE
}, f, indent=2)
sign_view.to_csv(paths["matrix_csv"])

print("\nSaved to:", OUT_DIR)
for k,p in paths.items(): print(f" - {k}: {p}")

if USE_COLAB:
    try:
        for p in paths.values():
            files.download(p)  # type: ignore
    except Exception:
        pass


Upload: indici_borsa_filtrati.csv  e  pesi_ritracciati.csv


Saving indici_borsa_filtrati.csv to indici_borsa_filtrati (1).csv
Saving pesi_ritracciati.csv to pesi_ritracciati (1).csv
Using: indici_borsa_filtrati (1).csv and pesi_ritracciati (1).csv


ValueError: Finestra troppo corta per iniziare.

In [ ]:
from google.colab import files

files.download('qlearning_outputs/weights_rl_OOS_split.csv')
files.download('qlearning_outputs/tilts_rl_OOS_split.csv')
files.download('qlearning_outputs/actions_rl_OOS_split.csv')
files.download('qlearning_outputs/portfolio_returns_rl_OOS_split.csv')
files.download('qlearning_outputs/metrics_OOS_split.json')
files.download('qlearning_outputs/overunder_matrix_Y.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>